# Module 04-2: 이메일 분류 에이전트 설계 및 구현

## 학습 목표

1. **State 직접 설계**: 이메일 분류에 필요한 `EmailClassifierState`를 스스로 설계한다
2. **4노드 구현**: parse_email → classify_email → assess_urgency → generate_action 흐름을 구현한다
3. **조건부 분기**: 스팸 이메일은 `assess_urgency`를 건너뛰는 분기를 구현한다
4. **3가지 테스트**: 정상 업무 이메일, 스팸, 빈 이메일을 모두 테스트한다

---
> 참고 문서: `docs/part2-first-agent/04-building-first-agent.md`

## 개념 요약

### 이메일 분류 에이전트 워크플로우

```
[시작]
  |
  v
[parse_email] ─── 이메일 파싱 및 유효성 검사
  |
  ├─ (빈 이메일) ──> [END] (에러 경로)
  |
  v
[classify_email] ─── LLM으로 스팸/정상 분류
  |
  ├─ (스팸) ──────────────────────────────────> [generate_action] ─> [END]
  |
  v (정상)
[assess_urgency] ─── LLM으로 긴급도 평가
  |
  v
[generate_action] ─── 액션 아이템 생성
  |
  v
[END]
```

### 설계 포인트
- 스팸 이메일은 긴급도 평가가 불필요 → `assess_urgency` 건너뜀
- 빈 이메일은 처리 불가 → 즉시 에러 종료
- LLM이 필요한 노드: classify_email, assess_urgency
- LLM이 불필요한 노드: parse_email, generate_action

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
from typing import TypedDict
from functools import partial
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: EmailClassifierState 설계 (TODO)

이메일 분류 에이전트에 필요한 State를 직접 설계하세요.

**설계 지침**:
- 입력 필드: 원본 이메일 텍스트
- 중간 필드: 파싱된 이메일 정보, 분류 결과, 긴급도 평가
- 출력 필드: 최종 액션 아이템
- 제어 필드: 현재 단계, 에러 메시지, 분류 결과(spam/ham)

**힌트**: 다음 정보가 필요합니다
- 이메일 원문
- 발신자, 제목, 본문 (파싱 후)
- 스팸 여부 판단 결과
- 긴급도 (high/medium/low)
- 추천 액션
- 현재 단계와 에러 메시지

In [ ]:
# TODO: EmailClassifierState를 직접 설계하세요

# 힌트 1: 이메일 처리에 필요한 데이터 흐름을 생각해보세요
#         raw_email → (파싱) → sender/subject/body → (분류) → category → (긴급도) → urgency → action
# 힌트 2: 분류 결과는 "spam" 또는 "ham" (정상) 문자열로 표현합니다
# 힌트 3:
# class EmailClassifierState(TypedDict):
#     raw_email: str             # 원본 이메일
#     sender: str | None        # 발신자
#     subject: str | None       # 제목
#     body: str | None          # 본문
#     category: str | None      # "spam" 또는 "ham"
#     urgency: str | None       # "high", "medium", "low"
#     action: str | None        # 추천 액션
#     current_step: str         # 현재 단계
#     error: str | None         # 에러 메시지

# EmailClassifierState를 정의하세요

print("EmailClassifierState 정의 완료")

In [ ]:
# 검증 셀
assert 'EmailClassifierState' in dir(), "EmailClassifierState가 정의되어야 합니다"
fields = list(EmailClassifierState.__annotations__.keys())
required = ['raw_email', 'category', 'urgency', 'action', 'current_step', 'error']
for f in required:
    assert f in fields, f"'{f}' 필드가 필요합니다. 현재: {fields}"
print(f"설계된 필드: {fields}")
print("✅ 실습 1 완료!")

## 실습 2: 4노드 구현 (TODO)

다음 4개의 노드 함수를 구현하세요.

**parse_email**:
- 이메일 텍스트에서 발신자(From:), 제목(Subject:), 본문을 파싱
- 빈 이메일이면 에러 반환

**classify_email**:
- LLM으로 스팸 여부 분류 ("spam" 또는 "ham" 반환)
- category 필드에 저장

**assess_urgency**:
- LLM으로 긴급도 평가 ("high", "medium", "low" 중 하나)
- urgency 필드에 저장

**generate_action**:
- category와 urgency를 기반으로 추천 액션 문자열 생성 (LLM 불필요)

In [ ]:
# TODO: parse_email 노드 구현

# 힌트 1: 줄 단위로 파싱하여 "From:", "Subject:" 접두사를 찾습니다
# 힌트 2: 빈 이메일은 에러를 반환합니다
# 힌트 3:
# def parse_email(state: EmailClassifierState) -> dict:
#     raw = state.get("raw_email", "")
#     if not raw or not raw.strip():
#         return {"error": "이메일이 비어있습니다.", "current_step": "error"}
#     lines = raw.strip().split("\n")
#     sender = ""
#     subject = ""
#     body_lines = []
#     in_body = False
#     for line in lines:
#         if line.startswith("From:"):
#             sender = line[5:].strip()
#         elif line.startswith("Subject:"):
#             subject = line[8:].strip()
#         elif in_body or (not line.startswith("From:") and not line.startswith("Subject:")):
#             in_body = True
#             body_lines.append(line)
#     body = "\n".join(body_lines).strip()
#     print(f"  [parse_email] 파싱 완료: sender='{sender}', subject='{subject}'")
#     return {"sender": sender, "subject": subject, "body": body, "current_step": "parsed"}

# parse_email 함수를 구현하세요

print("parse_email 구현 완료")

In [ ]:
# TODO: classify_email, assess_urgency, generate_action 노드 구현

# 힌트 1 (classify_email): LLM에 스팸 여부를 판단하게 하고, 응답에 "spam"이 포함되면 스팸 처리
# 힌트 2 (assess_urgency): LLM에 긴급도를 판단하게 하고, high/medium/low 중 하나를 추출
# 힌트 3 (generate_action): 스팸이면 "스팸 폴더로 이동", 긴급도에 따라 다른 액션 추천

# classify_email 함수를 구현하세요
# def classify_email(state: EmailClassifierState, llm=None) -> dict:
#     ...

# assess_urgency 함수를 구현하세요
# def assess_urgency(state: EmailClassifierState, llm=None) -> dict:
#     ...

# generate_action 함수를 구현하세요 (LLM 불필요)
# def generate_action(state: EmailClassifierState) -> dict:
#     category = state.get("category", "ham")
#     urgency = state.get("urgency", "medium")
#     if category == "spam":
#         action = "스팸 이메일로 분류됩니다. 스팸 폴더로 이동하고 발신자를 차단하세요."
#     elif urgency == "high":
#         action = "긴급 이메일입니다. 즉시 확인하고 2시간 이내에 답변하세요."
#     elif urgency == "medium":
#         action = "일반 업무 이메일입니다. 오늘 중으로 확인하고 처리하세요."
#     else:
#         action = "낮은 우선순위 이메일입니다. 시간 나는 대로 확인하세요."
#     return {"action": action, "current_step": "completed"}

print("4개 노드 구현 완료")

In [ ]:
# 검증 셀: 개별 노드 테스트
base_state = {
    "raw_email": "", "sender": None, "subject": None, "body": None,
    "category": None, "urgency": None, "action": None,
    "current_step": "start", "error": None
}

# parse_email 테스트
sample_email = "From: alice@example.com\nSubject: 미팅 일정 확인\n\n내일 오후 2시에 미팅이 있습니다."
r_parse = parse_email({**base_state, "raw_email": sample_email})
assert r_parse.get("sender") is not None, "sender가 파싱되어야 합니다"
assert r_parse.get("subject") is not None, "subject가 파싱되어야 합니다"

# parse_email 에러 케이스
r_empty = parse_email({**base_state, "raw_email": ""})
assert r_empty.get("error") is not None, "빈 이메일에서 error가 설정되어야 합니다"

print(f"파싱 결과: sender='{r_parse.get('sender')}', subject='{r_parse.get('subject')}'")
print("✅ parse_email 검증 완료!")

## 실습 3: 조건부 분기 및 그래프 조립 (TODO)

**분기 함수** 구현:
- `_route_after_parse(state)`: error 있으면 `"error"`, 없으면 `"classify"` 반환
- `_route_after_classify(state)`: category가 "spam"이면 `"action"`, 없으면 `"urgency"` 반환

**build_email_classifier_graph(llm)** 함수:
- 4개 노드 등록
- 조건부 엣지 2개: parse 이후, classify 이후
- 일반 엣지: assess_urgency → generate_action → END

In [ ]:
# TODO: 분기 함수와 그래프 빌드 함수를 구현하세요

# 힌트 1: 스팸이면 assess_urgency를 건너뛰고 바로 generate_action으로 이동합니다
# 힌트 2: add_conditional_edges의 딕셔너리 키와 분기 함수의 반환값이 일치해야 합니다
# 힌트 3:
# def _route_after_parse(state: EmailClassifierState) -> str:
#     return "error" if state.get("error") else "classify"
#
# def _route_after_classify(state: EmailClassifierState) -> str:
#     return "action" if state.get("category") == "spam" else "urgency"
#
# def build_email_classifier_graph(llm=None):
#     graph = StateGraph(EmailClassifierState)
#     graph.add_node("parse_email", parse_email)
#     graph.add_node("classify_email", partial(classify_email, llm=llm))
#     graph.add_node("assess_urgency", partial(assess_urgency, llm=llm))
#     graph.add_node("generate_action", generate_action)
#     graph.set_entry_point("parse_email")
#     graph.add_conditional_edges("parse_email", _route_after_parse, {"classify": "classify_email", "error": END})
#     graph.add_conditional_edges("classify_email", _route_after_classify, {"urgency": "assess_urgency", "action": "generate_action"})
#     graph.add_edge("assess_urgency", "generate_action")
#     graph.add_edge("generate_action", END)
#     return graph.compile()

# 분기 함수와 build_email_classifier_graph를 구현하세요

print("그래프 조립 함수 정의 완료")

In [ ]:
# 검증 셀: 분기 함수 테스트
s_ok = {"raw_email": "x", "sender": None, "subject": None, "body": None,
        "category": None, "urgency": None, "action": None, "current_step": "parsed", "error": None}
s_err = {**s_ok, "error": "에러"}
s_spam = {**s_ok, "category": "spam"}
s_ham = {**s_ok, "category": "ham"}

assert _route_after_parse(s_ok) == "classify", "error 없을 때 'classify'를 반환해야 합니다"
assert _route_after_parse(s_err) == "error", "error 있을 때 'error'를 반환해야 합니다"
assert _route_after_classify(s_spam) == "action", "스팸이면 'action'을 반환해야 합니다"
assert _route_after_classify(s_ham) == "urgency", "정상 메일이면 'urgency'를 반환해야 합니다"

print("✅ 분기 함수 검증 완료!")

## 실습 4: 3가지 테스트 시나리오

FakeLLM을 설정하고 다음 3가지 케이스를 테스트하세요:
1. 정상 업무 이메일
2. 스팸 이메일
3. 빈 이메일

In [ ]:
# FakeLLM 설정
email_llm = FakeLLM(
    responses={
        "스팸": "spam",
        "spam": "spam",
        "무료": "spam",
        "긴급": "high",
        "미팅": "ham",
        "분류": "ham",
    },
    default_response="ham",
)

# 긴급도 FakeLLM (분류 LLM과 별도)
urgency_llm = FakeLLM(
    responses={
        "긴급": "high",
        "즉시": "high",
        "미팅": "medium",
        "일반": "low",
    },
    default_response="medium",
)

graph = build_email_classifier_graph(llm=email_llm)
print("그래프 빌드 완료!")

In [ ]:
# 테스트 1: 정상 업무 이메일
normal_email = {
    "raw_email": "From: alice@company.com\nSubject: 내일 미팅 일정 확인\n\n안녕하세요. 내일 오후 2시 미팅 참석 가능하신지 확인 부탁드립니다.",
    "sender": None, "subject": None, "body": None,
    "category": None, "urgency": None, "action": None,
    "current_step": "start", "error": None
}

print("=" * 50)
print("테스트 1: 정상 업무 이메일")
print("=" * 50)
result1 = graph.invoke(normal_email)
print(f"분류: {result1.get('category')}")
print(f"긴급도: {result1.get('urgency')}")
print(f"액션: {result1.get('action')}")
print(f"단계: {result1.get('current_step')}")

In [ ]:
# 테스트 2: 스팸 이메일
spam_email = {
    "raw_email": "From: promo@spam.com\nSubject: 무료 상품 당첨 축하합니다!\n\n스팸 광고 내용입니다. 지금 바로 클릭하세요!",
    "sender": None, "subject": None, "body": None,
    "category": None, "urgency": None, "action": None,
    "current_step": "start", "error": None
}

print("=" * 50)
print("테스트 2: 스팸 이메일")
print("=" * 50)
result2 = graph.invoke(spam_email)
print(f"분류: {result2.get('category')}")
print(f"긴급도: {result2.get('urgency')} (스팸이므로 None이어야 함)")
print(f"액션: {result2.get('action')}")
print(f"단계: {result2.get('current_step')}")

In [ ]:
# 테스트 3: 빈 이메일
empty_email = {
    "raw_email": "",
    "sender": None, "subject": None, "body": None,
    "category": None, "urgency": None, "action": None,
    "current_step": "start", "error": None
}

print("=" * 50)
print("테스트 3: 빈 이메일")
print("=" * 50)
result3 = graph.invoke(empty_email)
print(f"에러: {result3.get('error')}")
print(f"단계: {result3.get('current_step')}")

In [ ]:
# E2E 검증 셀
# 테스트 1: 정상 이메일
assert result1.get("error") is None, f"정상 이메일에서 error가 없어야 합니다. 현재: {result1.get('error')}"
assert result1.get("action") is not None, "정상 이메일에서 action이 설정되어야 합니다"
assert result1.get("current_step") == "completed", f"정상 이메일에서 current_step이 'completed'여야 합니다. 현재: {result1.get('current_step')}"

# 테스트 2: 스팸 이메일
assert result2.get("error") is None, f"스팸 분류에서 error가 없어야 합니다. 현재: {result2.get('error')}"
assert result2.get("action") is not None, "스팸 이메일에서 action이 설정되어야 합니다"
# 스팸이면 assess_urgency를 건너뛰었으므로 urgency가 None이어야 함
if result2.get("category") == "spam":
    assert result2.get("urgency") is None, "스팸 이메일에서는 urgency가 평가되지 않아야 합니다"

# 테스트 3: 빈 이메일
assert result3.get("error") is not None, "빈 이메일에서 error가 설정되어야 합니다"
assert result3.get("current_step") == "error", f"빈 이메일에서 current_step이 'error'여야 합니다. 현재: {result3.get('current_step')}"

print("✅ 실습 4 완료! 3가지 테스트 시나리오 모두 통과")

## 정리

### 오늘 배운 내용
- **State 직접 설계**: 워크플로우에 맞는 State 필드를 스스로 결정하는 능력
- **스팸 건너뛰기 분기**: `_route_after_classify`로 스팸 이메일의 긴급도 평가를 건너뜀
- **3가지 테스트 패턴**: 정상/스팸/에러 케이스를 모두 테스트하는 습관
- **의존성 주입**: `functools.partial`로 LLM을 외부에서 주입하는 패턴 반복 연습

### 핵심 설계 원칙
- 불필요한 노드 실행을 조건부 분기로 건너뜀 → 효율성
- 에러는 가능한 빨리 감지하고 조기 종료 → 안정성
- LLM 의존성은 외부 주입 → 테스트 가능성

### 다음 모듈 안내
**Module 05: 프롬프트 엔지니어링** - 더 정확한 LLM 응답을 위한 프롬프트 설계 기법을 배웁니다.